# Aggregate elastic preds to find best hyper-params w/ MSE and r^2

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

BASE_SAVE_DIR = BASE_PATH / "initial_results"

## Aggregate all preds to find best params for each agg/impact/model/desc_seg combo

In [ ]:
import json

results_list = []
for agg_dir in (BASE_SAVE_DIR / "best_params").iterdir():
    agg_type = agg_dir.name
    print(agg_type)
    for impact_dir in agg_dir.iterdir():
        impact_name = impact_dir.name
        print(f"\t{impact_name}")
        for model_dir in impact_dir.iterdir():
            model_name = model_dir.name
            print(f"\t\t{model_name}")
            for desc_seg_path in model_dir.iterdir():
                # Desc if 0_shot, seg otherwise
                desc_seg_name = desc_seg_path.stem
                print(f"\t\t\t{desc_seg_name}")
                result_row = {
                    "agg": agg_type,
                    "impact": impact_name,
                    "model": model_name,
                    "desc_seg": desc_seg_name,
                }
                with open(desc_seg_path, "r") as file:
                    json_import = json.load(file)
                for metric_type in ["mse", "r2"]:
                    metric_dict = json_import[metric_type]
                    result_row = {
                        "agg": agg_type,
                        "impact": impact_name,
                        "model": model_name,
                        "desc_seg": desc_seg_name,
                        "metric_optimized": metric_type,
                        **metric_dict,
                    }
                    results_list.append(result_row)
result_df = pd.DataFrame(results_list)
result_df

### Export seperate tables for each metrics/impact combo 

In [ ]:
r2_df = result_df[result_df["metric_optimized"] == "r2"]

mse_df = result_df[result_df["metric_optimized"] == "mse"]
assert len(r2_df) + len(mse_df) == len(result_df)

In [ ]:
def order_export(df, optimize, export_dir):
    optimize_df = df[df["metric_optimized"] == optimize]
    df_dict = {
        "self": optimize_df[optimize_df["impact"] == "self"],
        "parent": optimize_df[optimize_df["impact"] == "parent"],
    }
    assert len(df_dict["self"]) + len(df_dict["parent"]) == len(optimize_df)
    other = "r2" if optimize == "mse" else "mse"
    ascending_bool = True if optimize == "mse" else False
    reorder_cols = [
        "agg",
        f"avg_{optimize}",
        f"stdev_{optimize}",
        f"avg_{other}",
        f"stdev_{other}",
        "l1_2",
        "alpha_2",
        "l1_3",
        "alpha_3",
        "model",
        "desc_seg",
        "metric_optimized",
        "impact",
    ]
    for impact, impact_df in df_dict.items():
        print(impact)
        ordered_df = impact_df[reorder_cols]
        ordered_df = ordered_df.sort_values(
            by=["agg", f"avg_{optimize}"],
            ascending=[True, ascending_bool],
            na_position="last",
        ).reset_index(drop=True)
        export_path = export_dir / f"{impact}.tsv"
        if export_path.exists():
            export_path.unlink()
        ordered_df.to_csv(export_path, sep="\t")
    return None

In [ ]:
for metric in ["r2", "mse"]:
    export_dir = BASE_PATH / "initial_results" / "best_params_summary" / metric
    export_dir.mkdir(exist_ok=True, parents=True)
    order_export(df=result_df, optimize=metric, export_dir=export_dir)

## Get r2 across 3 steps for best params of each impact/agg combo

- Will consider model_type and desc/seg type as hyper-params here

In [ ]:
from sklearn.metrics import r2_score

r2_analysis_rows = []
summary_dir = BASE_PATH / "initial_results" / "best_params_summary"
for impact_path in (summary_dir / "r2").iterdir():
    impact_name = impact_path.stem
    print(impact_name)
    metric_df = pd.read_csv(impact_path, sep="\t", index_col=0)
    for agg_type in metric_df["agg"].unique():
        print(agg_type)
        df_sub = metric_df[metric_df["agg"] == agg_type]
        # Ensure sorted
        df_sub_sorted = df_sub.sort_values(by="avg_r2", ascending=False).reset_index(
            drop=True
        )
        best_row = df_sub_sorted.iloc[0].to_dict()
        l1_2 = best_row["l1_2"]
        l1_3 = best_row["l1_2"]
        alpha_2 = best_row["alpha_2"]
        alpha_3 = best_row["alpha_3"]
        model_name = best_row["model"]
        desc_seg = best_row["desc_seg"]
        preds_dir = (
            BASE_PATH
            / "initial_preds"
            / agg_type
            / impact_name
            / model_name
            / desc_seg
            / f"alpha_{alpha_2}_{alpha_2}"
            / f"l1_{l1_2}_{l1_3}"
        )
        r2_l1_list = []
        r2_l2_list = []
        r2_l3_list = []
        for repeat_dir in preds_dir.iterdir():
            for split_path in repeat_dir.iterdir():
                preds_df = pd.read_parquet(split_path)
                y_true = preds_df["true"]
                r2_l1_list.append(
                    r2_score(y_true=y_true, y_pred=preds_df["pred_stage1"])
                )
                r2_l2_list.append(
                    r2_score(y_true=y_true, y_pred=preds_df["pred_stage2_total"])
                )
                r2_l3_list.append(
                    r2_score(y_true=y_true, y_pred=preds_df["pred_stage3_total"])
                )
        new_row = {
            "impact": impact_name,
            "agg": agg_type,
            "r2_layer1": np.mean(r2_l1_list),
            "r2_layer2": np.mean(r2_l2_list),
            "r2_layer3": np.mean(r2_l3_list),
        }
        r2_analysis_rows.append(new_row)
r2_analysis_df = pd.DataFrame(r2_analysis_rows)

In [ ]:
export_path = summary_dir / "r2_step_analysis.csv"
if export_path.exists():
    print(f"Removing path at {export_path}")
    export_path.unlink()
r2_analysis_df.to_csv(export_path, sep="\t")